# 📏 Chapter 5: Linear Models and Regularization
**Referensi Buku:** *scikit-learn Cookbook, Third Edition*

---
## 1. Pendahuluan
Model linier dasar seringkali terlalu menyesuaikan diri dengan *noise* pada data latih (*overfitting*). Bab ini mengeksplorasi teknik Regularisasi (Ridge, Lasso, Elastic Net) untuk membatasi kompleksitas model dan meningkatkan kemampuan generalisasinya pada data baru.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

%matplotlib inline
np.random.seed(42)

# Simulasi dataset dengan 50 fitur, TAPI hanya 10 fitur yang sebenarnya memiliki efek (informative)
X, y, true_coef = make_regression(n_samples=200, n_features=50, n_informative=10, 
                                  noise=15, coef=True, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Bentuk data latih: {X_train.shape}")
print(f"Jumlah fitur yang BENAR-BENAR informatif: {np.sum(true_coef != 0)}")

## 2. Ordinary Least Squares (Tanpa Regularisasi)
Mari kita lihat apa yang terjadi jika kita menggunakan regresi linier standar tanpa batas penalti.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

print(f"MSE Linear Regression pada Data Test: {mean_squared_error(y_test, lr.predict(X_test)):.2f}")
print(f"Jumlah fitur yang digunakan model (koefisien != 0): {np.sum(lr.coef_ != 0)}")

## 3. Ridge Regression (L2 Regularization)
Ridge Regression mencoba mengecilkan semua koefisien (shrinkage) untuk menghindari model yang terlalu percaya diri pada beberapa fitur saja. Kita menggunakan `RidgeCV` untuk mencari nilai `alpha` (kekuatan penalti) terbaik secara otomatis.

In [ ]:
from sklearn.linear_model import RidgeCV

# Mencari alpha terbaik di antara 0.1, 1.0, dan 10.0
ridge = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
ridge.fit(X_train, y_train)

print(f"Alpha terbaik yang ditemukan oleh RidgeCV: {ridge.alpha_}")
print(f"MSE Ridge Regression pada Data Test: {mean_squared_error(y_test, ridge.predict(X_test)):.2f}")
print(f"Jumlah fitur yang digunakan model (koefisien != 0): {np.sum(ridge.coef_ != 0)}")

# Ridge tidak pernah membuat koefisien tepat menjadi nol, hanya sangat kecil.

## 4. Lasso Regression (L1 Regularization)
Lasso (Least Absolute Shrinkage and Selection Operator) sangat kuat untuk dataset di mana banyak fitur sebenarnya *noise* (seperti dataset kita). Lasso akan menekan koefisien fitur yang tidak penting menjadi **tepat nol**.

In [ ]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train, y_train)

print(f"Alpha terbaik yang ditemukan oleh LassoCV: {lasso.alpha_:.4f}")
print(f"MSE Lasso Regression pada Data Test: {mean_squared_error(y_test, lasso.predict(X_test)):.2f}")
print(f"Jumlah fitur yang digunakan model (koefisien != 0): {np.sum(lasso.coef_ != 0)}")
print("Lasso berhasil membuang sebagian besar noise dan hanya menyimpan fitur yang relevan!")

## 5. Visualisasi Perbandingan Koefisien
Mari kita bandingkan secara visual nilai-nilai koefisien dari model Linear Regression biasa, Ridge, dan Lasso.

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(lr.coef_, 'o', label='Linear Regression', alpha=0.5, markersize=8)
plt.plot(ridge.coef_, '^', label='Ridge', alpha=0.7, markersize=8)
plt.plot(lasso.coef_, 's', label='Lasso', alpha=0.7, markersize=8)
plt.axhline(0, color='black', linestyle='--')

plt.xlabel('Indeks Fitur (0 - 49)')
plt.ylabel('Nilai Koefisien')
plt.title('Perbandingan Koefisien: Regresi Linier Standar vs Ridge vs Lasso')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 6. Elastic Net
Elastic Net menggabungkan sifat penalti L1 (dari Lasso) dan L2 (dari Ridge). Sangat berguna ketika Anda memiliki fitur-fitur yang berkorelasi sangat tinggi; di mana Lasso biasanya secara acak hanya memilih satu fitur dari kelompok yang berkorelasi, Elastic Net cenderung mempertahankan grup fitur tersebut bersama-sama.

In [ ]:
from sklearn.linear_model import ElasticNetCV

# l1_ratio adalah persentase efek L1. Jika l1_ratio=1, maka 100% Lasso.
elastic = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 1.0], cv=5, random_state=42)
elastic.fit(X_train, y_train)

print(f"L1_ratio terbaik: {elastic.l1_ratio_}")
print(f"Alpha terbaik: {elastic.alpha_:.4f}")
print(f"MSE Elastic Net pada Data Test: {mean_squared_error(y_test, elastic.predict(X_test)):.2f}")
print(f"Jumlah fitur yang digunakan: {np.sum(elastic.coef_ != 0)}")